# Driver Drowsiness Detection — CNN from Scratch

This notebook trains a **Convolutional Neural Network from scratch** (~9.8M parameters) on a labelled eye/face dataset to classify drivers as **Awake** or **Drowsy**.

## Workflow
1. Mount Google Drive / configure dataset path
2. Explore and visualise the dataset
3. Build the CNN architecture
4. Train with data augmentation
5. Evaluate on the test set
6. Save the model

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
import sys, os
# If running in Google Colab, mount Drive and add repo to path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Driver-Drowsiness-Detection-System'
except ImportError:
    REPO_ROOT = os.path.abspath('..')  # Running locally

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

from models.cnn_model import build_cnn_model, get_cnn_callbacks
from utils.data_loader import create_data_generators, DrowsinessDataLoader
from utils.metrics import (
    plot_training_history,
    plot_confusion_matrix,
    print_classification_report,
)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {len(tf.config.list_physical_devices("GPU"))}')

## 1. Configuration

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────
DATA_DIR     = 'data/'       # Change to your dataset path
IMAGE_SIZE   = (64, 64)
BATCH_SIZE   = 32
EPOCHS       = 50
SEED         = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

## 2. Dataset Exploration

In [ ]:
train_gen, val_gen, test_gen = create_data_generators(
    DATA_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

print(f'Training samples  : {train_gen.n}')
print(f'Validation samples: {val_gen.n}')
print(f'Test samples      : {test_gen.n}')
print(f'Class mapping     : {train_gen.class_indices}')

In [ ]:
# Visualise a batch
images, labels = next(train_gen)
class_names = ['Awake', 'Drowsy']

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for ax, img, lbl in zip(axes.ravel(), images, labels):
    ax.imshow(img)
    ax.set_title(class_names[int(lbl)], fontsize=8)
    ax.axis('off')
plt.suptitle('Training samples (with augmentation)', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Build CNN Model

In [ ]:
model = build_cnn_model(input_shape=(*IMAGE_SIZE, 3))
model.summary()
print(f'\nTotal parameters: {model.count_params():,}')

## 4. Train

In [ ]:
Path('saved_models').mkdir(exist_ok=True)
callbacks = get_cnn_callbacks('saved_models/cnn_best.weights.h5')

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
plot_training_history(history, model_name='CNN from Scratch')

## 5. Evaluate on Test Set

In [ ]:
test_gen.reset()
y_pred_proba = model.predict(test_gen, verbose=1).ravel()
y_true = test_gen.classes

metrics = print_classification_report(y_true, y_pred_proba, model_name='CNN from Scratch')
print('\nKey metrics:', metrics)

In [ ]:
plot_confusion_matrix(
    y_true, (y_pred_proba >= 0.5).astype(int),
    model_name='CNN from Scratch'
)

## 6. Save Model

In [ ]:
model.save('saved_models/cnn_final.keras')
print('Model saved to saved_models/cnn_final.keras')